In [ ]:
import cv2
import numpy as np
from morph import *

mm.install()

def processar_frame(frame):
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    blur = cv2.GaussianBlur(gray, (15, 15), 0)
    
    _, binaria = cv2.threshold(blur, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
    
    binaria = mm.open(binaria, mm.sedisk(5))
    
    dist = mm.dist(binaria)
    _, sure_fg = cv2.threshold(dist, 0.7 * dist.max(), 255, 0)
    sure_fg = np.uint8(sure_fg)
    
    markers = mm.label0(sure_fg)
    markers = np.uint8(markers)
    
    segmentada = mm.watershed(frame, markers)
    
    output = frame.copy()
    labels = np.unique(segmentada)
    
    for label in labels:
        if label <= 0: continue
        
        mask = np.uint8(segmentada == label) * 255
        cnts, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        
        if cnts:
            c = max(cnts, key=cv2.contourArea)
            if cv2.contourArea(c) > 500: # Filtro para ignorar artefatos pequenos
                cv2.drawContours(output, [c], -1, (0, 255, 0), 2)
                
                M = cv2.moments(c)
                if M["m00"] != 0:
                    cX, cY = int(M["m10"]/M["m00"]), int(M["m01"]/M["m00"])
                    cv2.putText(output, f"ID {label}", (cX-20, cY), 
                                cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)
    return output

# Captura de Vídeo
cap = cv2.VideoCapture(0)

while True:
    ret, frame = cap.read()
    if not ret: break
    
    resultado = processar_frame(frame)
    
    cv2.imshow('PDI - Segmentacao Watershed Real-Time', resultado)
    
    if cv2.waitKey(1) & 0xFF == ord('q'): # Pressione 'q' para sair
        break

cap.release()
cv2.destroyAllWindows()